# THX-01 — phone/edge-scale calibrated decisions on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/THX-01-decision-routing.ipynb)

Run HAL-X AI's [`doofz/THX-01`](https://huggingface.co/doofz/THX-01) locally with `pip install thx01`:
give it a state (a ticket, email, JSON record, or document) plus typed questions, and get **calibrated
probabilities back from one forward pass** — no generated tokens, no chat template.

THX-01 is a **322M decision model** (307M [mmBERT-base](https://huggingface.co/jhu-clsp/mmBERT-base) encoder
+ 15M decision head), post-trained with Reinforcement Learning for Calibrated Decisions (RLCD). It is
**not a chat model**: it does not write text. It is built for phone- and edge-scale routing, triage,
priority scoring, and document lookup — the same *kind* of job as
[`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb), at about **one-tenth the size**,
text-only, and without a training loop (the Unsloth email-triage notebooks train a 4B decision head).

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU* is the fastest path. The model is tiny
(~644 MB of weights, about 0.65 GB in float16), so a **CPU runtime also works**. The T4 has no native
bfloat16 and no FlashAttention-2; this notebook does not ask for either. `thx01` picks CUDA when
present and stores the weights in float16 on a T4 (compute capability < 8); on CPU it uses float32.

**Links:** [model card](https://huggingface.co/doofz/THX-01) ·
[blog](https://huggingface.co/blog/doofz/thx-01-we-open-sourced-a-322m-decision-model-that) ·
[demo](https://huggingface.co/spaces/doofz/THX-01-demo) ·
[hosted API](https://api.hal-x.ai/docs/thx-01)


In [ ]:
%pip install -q -U thx01

# The card and blog install is `pip install thx01`. Weights download from
# https://huggingface.co/doofz/THX-01 on first load (~644 MB).
# Dependencies (torch, transformers>=4.48, safetensors, huggingface_hub, numpy)
# are already on Colab. No FlashAttention-2 extra, no bitsandbytes, no Unsloth.


> Restart is usually unnecessary: `thx01` is a new package, not an upgrade of a library Colab already
> imported. If the install cell upgraded `transformers` and a later cell errors on tokenizer config,
> use *Runtime > Restart session* and continue from the next cell.


## 1. Load THX-01 and print size

`thx01.load("doofz/THX-01")` uses GPU if one is attached. First run downloads the snapshot
(`config.json`, `model.safetensors`, `tokenizer/`, `encoder/`).


In [ ]:
import time

import thx01
import torch

print("thx01", thx01.__version__)
print("torch", torch.__version__)

t0 = time.perf_counter()
agent = thx01.load("doofz/THX-01")
load_s = time.perf_counter() - t0

n_params = sum(p.numel() for p in agent.model.parameters())
print(f"loaded in {load_s:.1f}s")
print(f"device: {agent.device} | dtype: {agent.dtype}")
print(f"parameters: {n_params / 1e6:.1f}M  (card: 322M = 307M encoder + 15M head)")
if agent.device.type == "cuda":
    name = torch.cuda.get_device_name()
    cap = torch.cuda.get_device_capability()
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"GPU: {name}  sm_{cap[0]}{cap[1]}  | VRAM allocated: {allocated:.2f} GB")
    print("latency tip: the card measures ~10 ms / request on one GPU, ~1 ms / decision when batched.")
else:
    print("latency tip: CPU is fine for this 322M model; expect ~200–500 ms / request, not ~10 ms.")
    print("pick Runtime > Change runtime type > T4 GPU if you want the card's ~10 ms path.")


## 2. Question types and a print helper

`agent.decide(state, questions)` is the public API (the HTTP `/v1/decide` shape). Each question needs
a `type` and the question text. The card accepts `question` or `instructions`; this notebook uses
`instructions` so the same dicts work on both `decide()` and `predict_batch()`. `choice` and `score`
also take `criteria`.

| `type` | `criteria` | answer |
|---|---|---|
| `choice` | `{name: description}` or a list | `choice`, `probabilities`, `confidence` |
| `noul` | optional | `noul` = P(yes), `confidence` |
| `score` | ordered list of levels, lowest first | `score` (expected level), `legend`, `probabilities` |
| `number` | optional `min` / `max` / `precision` | a value **stated in the document**, or `null` |
| `excerpt` | none | a verbatim span with character offsets |
| `"cite": true` | on any question | supporting parts of the document |

`number`, `excerpt`, and citations are native THX-01 types. They are not in TypeSafe / d1-3B
(`choice` / `noul` / `score` only).


In [ ]:
import json


def show(result, title=None):
    if title:
        print(title)
    answers = result["answers"] if isinstance(result, dict) and "answers" in result else result
    for name, a in answers.items():
        kind = a.get("type")
        if kind == "choice":
            probs = ", ".join(f"{k} {p:.2f}" for k, p in a["probabilities"].items())
            print(f"{name:>16}: {a['choice']}  ({a['confidence']:.2f})  [{probs}]")
        elif kind == "noul":
            print(f"{name:>16}: P(yes) = {a['noul']:.3f}  (conf {a['confidence']:.2f})")
        elif kind == "score":
            level = int(round(a["score"]))
            label = a["legend"].get(str(level), "")
            print(f"{name:>16}: {a['score']:.2f} -> {label!r}  (conf {a['confidence']:.2f})")
        elif kind == "number":
            print(f"{name:>16}: {a['value']}  (p={a['probability']:.2f}, not_stated={a['not_stated']:.2f})")
        elif kind == "excerpt":
            text = a.get("text")
            print(f"{name:>16}: {text!r}  (p={a.get('probability', 0):.2f})")
        else:
            print(f"{name:>16}: {a}")
        if a.get("citations"):
            for c in a["citations"]:
                print(f"{'':>16}  cite p={c['probability']:.2f}: {c['text'][:120]}")
    usage = result.get("usage") if isinstance(result, dict) else None
    if usage:
        print(f"{'usage':>16}: {usage}")


# Warm-up: the first shape is slower (CUDA kernels, CPU oneDNN). Time the next cells after this.
_ = agent.decide(
    "Please refund the duplicate charge on invoice 4412.",
    {"team": {"type": "choice", "instructions": "Which team?", "criteria": {"billing": "charges", "other": "other"}}},
)
print("warm-up done")


## 3. Intent / routing in one pass

Several typed questions about the same ticket, one `decide()` call. Labels are supplied at inference
time — no fine-tune, no prompt template.


In [ ]:
ticket = (
    "I was charged twice for the same Pro plan this month. Invoice #18442 shows two $29 hits. "
    "Please refund one of them today — payroll closes Friday and I need the statement to match."
)

routing = {
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this support ticket?",
        "criteria": {
            "billing": "Charges, refunds, invoices, failed payments",
            "technical": "App crashes, bugs, outages, integrations",
            "shipping": "Deliveries, tracking, returns of physical goods",
            "account": "Login, password, profile, email change",
            "security": "Unauthorised access, phishing, fraud on the account",
        },
    },
    "intent": {
        "type": "choice",
        "instructions": "What does the customer want?",
        "criteria": {
            "refund": "money back or a duplicate charge reversed",
            "status": "where is my order / what is the status",
            "fix": "something is broken and they want it working",
            "info": "a how-to or policy question",
            "cancel": "cancel or downgrade the plan",
        },
    },
    "refund": {
        "type": "noul",
        "instructions": "Is the customer asking for a refund or charge reversal?",
    },
}

t0 = time.perf_counter()
result = agent.decide(ticket, routing)
elapsed_ms = (time.perf_counter() - t0) * 1000
show(result, title=ticket)
print(f"{'latency':>16}: {elapsed_ms:.0f} ms")


## 4. Priority / severity score

`score` is an ordinal: the model returns a distribution over the ordered levels and an expected
value. Use it for urgency, frustration, or incident severity — not as a generated 1–5 rating.


In [ ]:
severity = {
    **routing,
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this ticket?",
        "criteria": [
            "can wait until next week",
            "should be handled today",
            "blocking the customer right now",
        ],
    },
    "frustration": {
        "type": "score",
        "instructions": "How frustrated does the customer sound?",
        "criteria": [
            "calm and neutral",
            "concerned but civil",
            "clearly annoyed",
            "angry or using strong language",
        ],
    },
}

show(agent.decide(ticket, severity))


## 5. Multilingual routing

THX-01 was post-trained in 18 languages (Azerbaijani-first) and inherits mmBERT's 100+ language
coverage. The same English label set works on tickets written in another language — including
transliteration and missing diacritics, which the card trains for.


In [ ]:
team_only = {"team": routing["team"]}

tickets = [
    ("en", "The app crashes every time I open Settings after yesterday's update."),
    ("az", "Kartımdan eyni ödəniş iki dəfə çıxıb, zəhmət olmasa birini geri qaytarın."),
    ("az-ascii", "Kartimdan eyni odenis iki defe cixib, birini geri qaytarin."),  # no ə/ş/ç
    ("tr", "Siparişim Pazartesi teslim edilecekti, hâlâ yolda görünüyor. Nerede?"),
    ("ru", "Меня дважды списали за одну и ту же подписку, верните один платёж."),
    ("fr", "L'application se ferme dès que j'ouvre les paramètres après la mise à jour."),
    ("de", "Die Lieferung sollte gestern ankommen. Wo ist das Paket?"),
    ("es", "Hay tres cargos en mi tarjeta que yo no hice. Bloqueen la cuenta."),
]

t0 = time.perf_counter()
batch = agent.predict_batch([text for _, text in tickets], [team_only] * len(tickets))
elapsed_ms = (time.perf_counter() - t0) * 1000

for (lang, text), row in zip(tickets, batch):
    a = row["answers"]["team"]
    print(f"{lang:>8}  {a['choice']:<10} ({a['confidence']:.2f})  {text}")
print(f"\n{len(tickets)} tickets in {elapsed_ms:.0f} ms  ({elapsed_ms / len(tickets):.0f} ms/ticket)")


## 6. Calibrated probabilities as a policy

RLCD trains on a strictly proper scoring rule, so a reported 0.97 should be right about 97% of the
time on similar decisions (card ECE 0.003 on the held-out ticket set). That turns a threshold into
an operating policy: auto-route the confident ones, escalate the rest.

The card's selective-automation figure: about 75% of tickets handled automatically with no error on
their four sets, and 90% of tickets at ≥ 99.5% accuracy.


In [ ]:
AUTO_MIN = 0.90  # raise toward 0.99 for a stricter auto-route band

policy_tickets = [
    "Please email me last month's invoice as a PDF.",
    "I was charged twice, refund the extra $29.",
    "hmm not sure if this is a billing thing or the app is just broken? payment page spins forever",
    "There are three purchases on my card I never made. Lock the account.",
    "hi",
]

print(f"{'action':<10} {'team':<10} {'p':>5}  ticket")
print("-" * 88)
for text in policy_tickets:
    a = agent.decide(text, team_only)["answers"]["team"]
    p = a["confidence"]
    action = "AUTO" if p >= AUTO_MIN else "ESCALATE"
    print(f"{action:<10} {a['choice']:<10} {p:5.2f}  {text}")

print()
print("Full distribution on the ambiguous ticket:")
print(json.dumps(agent.decide(policy_tickets[2], team_only)["answers"]["team"]["probabilities"], indent=2))


## 7. Native number, excerpt, and citation (optional)

d1-3B (and TypeSafe `choice` / `noul` / `score`) cannot return a stated dollar amount or a verbatim
quote without extra service-layer calls. THX-01 can. `number` copies a value that is **in the
document** (no arithmetic, no currency conversion). `excerpt` cuts a span out of the text, so it
cannot invent wording.


In [ ]:
doc = (
    "Northwind Corp. reported Q3 2026 results. Revenue for the quarter was $48.3 million, up 14%. "
    "Net income was $6.2 million, compared with $4.1 million a year ago. "
    '"We will open two offices in Baku," said CEO Laura Chen.'
)

extract = {
    "kind": {
        "type": "choice",
        "instructions": "What is this document?",
        "criteria": {
            "earnings": "earnings report",
            "complaint": "customer complaint",
            "invoice": "invoice",
        },
    },
    "revenue": {
        "type": "number",
        "instructions": "What was the revenue, in US dollars?",
        "cite": True,
    },
    "quote": {
        "type": "excerpt",
        "instructions": "What did the CEO say?",
    },
}

extracted = agent.decide(doc, extract)
show(extracted, title=doc)
print()
print(json.dumps(extracted["answers"]["revenue"], indent=2, default=str)[:800])


## 8. Why THX-01 vs d1-3B (without loading d1)

This cell is a comparison, not a bake-off. It does **not** download
[`LiquidAI/d1-3B`](https://huggingface.co/LiquidAI/d1-3B). Use
[`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb) when you need vision.

| | **THX-01** (this notebook) | **d1-3B** (the other decision notebook) |
|---|---|---|
| Size | **322M** (~0.65 GB float16) | 3.1B (~6.2 GB float16) |
| Encoder | mmBERT-base (text) | LFM2.5-VL-3B (text + **images**) |
| Modalities | text / JSON / agent traces | text / JSON / **images** |
| Question types | `choice`, `noul`, `score`, plus native **`number` / `excerpt` / cite** | `choice`, `noul`, `score` |
| Languages | 18 post-trained (Azerbaijani-first), 100+ via mmBERT | multilingual VL checkpoint |
| Latency (card) | ~10 ms / request on one GPU | one forward pass, larger model |
| Fits | phone / edge / **free T4 or CPU** | free T4 (~7 GB peak with an image) |
| Library | `pip install thx01` → `agent.decide` | `transformers` `AutoModel` → `model.system_one` |
| Licence | **Apache 2.0** | LFM Open License v1.0 |

Reach for THX-01 when the state is text and you want phone-scale latency or native lookup.
Reach for d1-3B when the state is a picture (or you already standardised on System One in `transformers`).


## Caveats and licence

- **Not a chat model.** It does not generate replies, explanations, or tool calls. Ask typed
  questions; read structured answers.
- **Fine-grained overlapping labels stay hard.** Banking77 (77 near-synonymous intents) is 65.9%
  on the card; DAIR Emotion (6 overlapping classes) is 44.6%, with correspondingly low confidence.
- **`number` and `excerpt` locate text that is already there.** They do not add numbers, convert
  units or currencies, or paraphrase. Ask for kilometres when the document states miles and you
  get `null`.
- **Context is short.** Each question (question + options + state) is truncated at 1,024 tokens.
- **Calibration is not a guarantee on your labels.** The card's ECE is on their ticket and suite
  evals. Re-measure on your traffic before you set an auto-route threshold.
- **Licence:** [Apache License 2.0](https://huggingface.co/doofz/THX-01/blob/main/LICENSE).
  See also [`NOTICE`](https://huggingface.co/doofz/THX-01/blob/main/NOTICE).

**Next steps:** swap in your own `criteria`; raise `AUTO_MIN` once you have a labelled sample;
serve with `pip install "thx01[server]"` (`POST /v1/decide`) if you want the REST shape from the card.
